In [18]:
import os
from pathlib import Path
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split, cross_validate
from sklearn.metrics import ConfusionMatrixDisplay
import shap

# --- Configuration (change if you want) ---
RANDOM_STATE = 42
RF_N_ESTIMATORS = 200
RF_N_JOBS = -1
CV_N_SPLITS = 25
TEST_SIZE = 0.2
MAX_DISPLAY_BEESWARM = 40
MAX_DISPLAY_IMPORTANCES = 20
# -------------------------------------------

def train_and_evaluate(rfc, X, y, y_uniques, fn):
    sss = StratifiedShuffleSplit(n_splits=CV_N_SPLITS, test_size=TEST_SIZE)
    sss_scores = cross_validate(rfc, X, y, cv=sss)
    print(f"Mean fit time: {sss_scores['fit_time'].mean():.2f}s")
    print(f"Accuracy mean: {sss_scores['test_score'].mean():.2%}, median: {np.median(sss_scores['test_score']):.2%}\n")

    # held-out deterministic test split for confusion matrix
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, random_state=RANDOM_STATE, test_size=TEST_SIZE, stratify=y
    )

    # fit on X_train for evaluation
    rfc.fit(X_train, y_train)

    # confusion matrix on the held-out test
    ConfusionMatrixDisplay.from_estimator(
        rfc, X_test, y_test, display_labels=list(y_uniques)
    )

    title = (
        f"{Path(fn).name}\n"
        f"Mean fit time: {sss_scores['fit_time'].mean():.2f}s\n"
        f"Accuracy mean: {sss_scores['test_score'].mean():.2%}, median: {np.median(sss_scores['test_score']):.2%}\n"
    )
    plt.title(title, loc="left")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(f"{Path(fn).stem}_confusion.png", bbox_inches="tight")
    plt.close()

    # Re-fit on full data for final explanations
    rfc_full = RandomForestClassifier(
 #       n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=RF_N_JOBS
    )
    rfc_full.fit(X, y)

    return rfc_full


def compute_shap_for_model(rfc, X):
    explainer = shap.TreeExplainer(rfc)
    shap_vals = explainer.shap_values(X)

    if isinstance(shap_vals, list):
        shap_arr = np.stack(shap_vals, axis=2)  # (n_samples, n_features, n_classes)
    else:
        shap_arr = shap_vals  # keep as (n_samples, n_features) for binary/regression
    return shap_arr, list(X.columns)

def plot_beeswarms(fn, shap_arr, feature_names, y_uniques, X):
    n_classes = shap_arr.shape[2] if shap_arr.ndim == 3 else 1
    for class_idx, target_class in enumerate(y_uniques):
        if n_classes > 1:
            vals = shap_arr[..., class_idx]  # shape (n_samples, n_features)
        else:
            vals = shap_arr  # binary case

        expl = shap.Explanation(
            values=np.array(vals),  # ensure shape (n_samples, n_features)
            data=X.values,
            feature_names=feature_names,
        )

        shap.plots.beeswarm(expl, max_display=MAX_DISPLAY_BEESWARM, show=False)
        plt.title(f"{Path(fn).name}\nSHAP Beeswarm, class: {target_class}\n", loc="left")
        plt.savefig(f"{Path(fn).stem}_beeswarm_{target_class}.png", bbox_inches="tight")
        plt.tight_layout()
        plt.close()


def plot_singleclass_importances(fn, shap_arr, feature_names, y_uniques, X):
    n_classes = shap_arr.shape[2]
    for class_idx, target_class in enumerate(y_uniques):
        shap.plots.bar(
            shap_arr[..., class_idx],
            features=X,
            max_display=MAX_DISPLAY_IMPORTANCES,
            show=False,
        )
        plt.title(f"{Path(fn).name}\nSHAP single-class importances, class: {target_class}\n", loc="left")
        plt.savefig(f"{Path(fn).stem}_importances_{target_class}.png", bbox_inches="tight")
        plt.tight_layout()
        plt.close()


def plot_multiclass_importances(fn, shap_arr, X, y_uniques):
    n_classes = shap_arr.shape[2]
    vals = [shap_arr[:, :, i] for i in range(n_classes)]
    shap.summary_plot(
        vals,
        X,
        class_names=list(y_uniques),
        plot_type="bar",
        show=False,
        max_display=MAX_DISPLAY_IMPORTANCES,
    )
    plt.title(f"{Path(fn).name}\nMulti-class feature importance\n", loc="left")
    plt.savefig(f"{Path(fn).stem}_importance_multiclass.png", bbox_inches="tight")
    plt.tight_layout()
    plt.close()


def do_plots(fn: str):
    import random
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    df = pd.read_csv(fn)
    X = df.iloc[:, 2:].copy()
    y, y_uniques = df.class_label.factorize()

    rfc = RandomForestClassifier(
        #n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=RF_N_JOBS
        )
    rfc_full = train_and_evaluate(rfc, X, y, y_uniques, fn)

    shap_arr, feature_names = compute_shap_for_model(rfc_full, X)

    # Pass X into the beeswarm / singleclass plotting functions
    plot_beeswarms(fn, shap_arr, feature_names, y_uniques, X)
#    plot_singleclass_importances(fn, shap_arr, feature_names, y_uniques, X)
#    plot_multiclass_importances(fn, shap_arr, X, y_uniques)

    print(f"Saved plots for {Path(fn).name} to current directory.")





In [21]:
for fn in glob.glob("../../data/dump_features/de/*.csv"):
    do_plots(fn)

Mean fit time: 0.20s
Accuracy mean: 99.80%, median: 100.00%

Saved plots for de_foot_token_1.csv to current directory.
Mean fit time: 0.26s
Accuracy mean: 78.90%, median: 80.00%

Saved plots for de_meter_token_2.csv to current directory.
Mean fit time: 0.21s
Accuracy mean: 100.00%, median: 100.00%

Saved plots for de_foot_pos_syl_2.csv to current directory.
Mean fit time: 0.31s
Accuracy mean: 95.20%, median: 95.00%

Saved plots for de_meter_pos_syl_2.csv to current directory.
Mean fit time: 0.29s
Accuracy mean: 82.10%, median: 82.50%

Saved plots for de_meter_token_1.csv to current directory.
Mean fit time: 0.18s
Accuracy mean: 98.80%, median: 100.00%

Saved plots for de_foot_pos_1.csv to current directory.
Mean fit time: 0.19s
Accuracy mean: 98.20%, median: 100.00%

Saved plots for de_foot_token_2.csv to current directory.
Mean fit time: 0.18s
Accuracy mean: 100.00%, median: 100.00%

Saved plots for de_foot_syl_2.csv to current directory.
Mean fit time: 0.19s
Accuracy mean: 88.60%, me

In [11]:
print(shap.__version__)

0.46.0
